In [37]:
import requests
import os
import json
import gradio as gr

from langchain_groq import ChatGroq

from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

from dotenv import load_dotenv

from typing import Any, Dict, List, Optional

import chromadb
from chromadb.utils import embedding_functions
from pypdf import PdfReader

In [38]:
key = os.getenv('assg_groq_key')
model = ChatGroq(model='qwen/qwen3.6-27b', temperature=0, api_key = key)

In [ ]:
# Model Configuration
_model = os.environ.get("model", "qwen/qwen3.6-27b")

# Controls how many times the agent can think and use tools. It prevents infinite loops where the agent keeps calling tools
Max_Steps = 5

system_prompt = """You are a helpful Travel Planning Assistant.

You have access to several tools for 
- Checking weather for a destination
- Converting currency using live exchange rates
- calculating trip budgets
- Searching the web for general travel information (e.g., visa requirements etc)
- Retrieving information from the user's own uploaded travel documents (e.g., visa documents)

Guidelines:
- Choose the tool(s) based on the user's request.
- If a question requires mutiple tools, use all the relevant ones before answering.
- Combine the results of the tools into one clear, well-organized final answer.
- If a tool cannot retrieve the requested information or returns an error, explain the problem in simple language and, if possible, suggest an alternative.
- If the user's question can be answered without using a tool, respond directly.
- Keep your responses friendly, accurate, and helpful.
"""

# Groq Client Initialization
_groq_client = None

def _get_client():
    global _groq_client
    
    if _groq_client is None:
        from groq import Groq
        api_key = os.getenv("assg_groq_key")
        
        # If API key is missing, stop execution
        if not api_key:
            raise RuntimeError("GROQ_API_KEY is not set.")
        
        # Create Groq API client
        _groq_client = Groq(api_key= api_key)
    return _groq_client

#TOOL 1: WEATHER TOOL

geo_url = "https://geocoding-api.open-meteo.com/v1/search"
weather_forecast_url = "https://api.open-meteo.com/v1/forecast"

@tool("get_weather")
def get_weather(city: str):
    #The description below helps LLM to understand the function and when should this tool be used.
    """
    Get the current weather for given city.
    Use this whenever the user asks about weather or temperature for travelling.
    """
    try:
        # Step 1: Get the city name and turn them into coordinates
        geo_match = requests.get(geo_url, 
                                 params={"name": city, "count": 1}, 
                                 timeout=10).json()

        #Check if city exists
        if not geo_match.get("results"):
            return f"Could not find a location matching '{city}'. Please check the spelling."

        #Get the first matching city
        location = geo_match["results"][0]

        #Extract coordinates
        latitude = location["latitude"]
        longitude = location["longitude"]

        # Step 2: get the current weather for those coordinates
        weather_forecast = requests.get(weather_forecast_url, 
                                        params={"latitude": latitude, "longitude": longitude, "current_weather": True},
                                        timeout=10).json()

        #Extract current weather
        current = weather_forecast["current_weather"]
        return f"Current weather in {location['name']}: {current['temperature']}°C, wind {current['windspeed']} km/h."

    #Handle network error
    except requests.exceptions.RequestException as e:
        return f"Weather API request failed: {e}"
    
    
# TOOL 2 : CURRENCY EXCHANGE API

@tool
def currency_converter(query: str) -> str:
    """
    Convert currency.
    Example: 100 USD to AUD
    """

    try:
        # Step 1: Clean and split user input
        amount, from_currency, to_currency = (query.upper().replace(" TO ", " ").split())

        # Convert amount from string to number
        amount = float(amount)
        
        # Step 2: Call Exchange Rate API
        url = f"https://api.exchangerate-api.com/v4/latest/{from_currency}"

        data = requests.get(url, timeout=10).json()

        # Step 3: Extract exchange rate
        rate = data["rates"][to_currency]

        # Step 4: Calculate converted amount
        result = amount * rate

        return (f"{amount} {from_currency} = "f"{result:.2f} {to_currency}")

    # Step 6: Handle errors
    except Exception as e:
        return f"Error: {e}"
    
    
# TOOL 3 : BUDGET CALCULATOR (CUSTOM PYTHON FUNCTION)

@tool("calculate_trip_budget")
def calculate_trip_budget(
    no_of_days: int,
    no_of_tourists: int,
    flight_cost_per_person: float,
    hotel_cost_per_night: float,
    food_cost_per_day: float,
    activities_cost_per_day: float,
    misc_percent: float = 10.0):
    
    #The description below helps LLM to understand the function and when should this tool be used.
    """
    Estimate the total cost of a trip given the number of days and average
    daily costs. 
    Use this whenever the user asks for a trip budget, cost
    estimate, or "how much will this trip cost".
    """
    
    try:
        if no_of_days <= 0 or no_of_tourists <= 0:
            return "Number of days must be greater than zero."

        #Formulas to calculate cost of hotel, food, flight and acitivities
        flight_cost = flight_cost_per_person * no_of_tourists
        hotel_cost = no_of_days * hotel_cost_per_night
        food_cost = no_of_days * food_cost_per_day * no_of_tourists
        activities_cost = no_of_days * activities_cost_per_day * no_of_tourists

        subtotal = flight_cost + hotel_cost + food_cost + activities_cost
        misc_total = subtotal * (misc_percent / 100.0)
        grand_total = subtotal + misc_total

        #Builds a report for the user
        breakdown = (
            f"Trip Budget Estimate = {no_of_days} day(s), {no_of_tourists} traveler(s):\n"
            f"- Hotel: {hotel_cost:.2f}\n"
            f"- Food: {food_cost:.2f}\n"
            f"- Activities: {activities_cost:.2f}\n"
            f"- Flight: {flight_cost:.2f}\n"
            f"- Miscellaneous ({misc_percent:.0f}%): {misc_total:.2f}\n"
            f"- Total estimated cost: {grand_total:.2f}"
        )
        return breakdown
    
    #Handle invalid inputs
    except (TypeError, ValueError) as e:
        return f"Could not calculate budget, please check the numbers provided: {e}"
    

# TOOL 4 : WEB SEARCH

_search = DuckDuckGoSearchRun()

@tool("web_search", return_direct=False)
def web_search(query: str):
  
  #The description below helps LLM to understand the function and when should this tool be used.
  """
    Search the web for up-to-date, general travel information that is not
    covered by the other tools like 
    - visa requirements
    - local events
    - safety advisories or
    - "what is X known for".
    """
  try:
      #Step 1: Perform web search
      results = _search.run(query)

      #Step 2: Checks whether search returned anything
      if not results:
          return "No search results were found for that query."
      
      #Step 3: Return search results
      return results
  
  #Handle errors
  except Exception as e:
      return f"Web search failed: {e}"
  
def search_documents(query: str):
    try:
        chunks = retrieve_docs(query)
        if not chunks:
            return "No relevant indexed documents found. (Has a PDF been uploaded yet?)"
        return "\n---\n".join(chunks)
    except Exception as e:
        return f"Document search failed: {e}"

# dictionary that connects LLM tool names with actual functions
tool_functions = {
    "get_weather": get_weather,
    "convert_currency": convert_currency,
    "web_search": web_search,
    "calculate_trip_budget": calculate_trip_budget,
    "search_documents": search_documents}

# This tells Groq  LLM - What tools exists, what each tool does and what parameters are required
tools_schema: List[Dict[str, Any]] = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current weather for a given city.",
            "parameters": {
                "type": "object",
                "properties": {"city": {"type": "string"}},
                "required": ["city"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "convert_currency",
            "description": "Convert an amount from one currency to another.",
            "parameters": {
                "type": "object",
                "properties": {
                    "amount": {"type": "number"},
                    "from_currency": {"type": "string"},
                    "to_currency": {"type": "string"},
                },
                "required": ["amount", "from_currency", "to_currency"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "web_search",
            "description": "Search the web for up-to-date travel information.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string"}
                },
                "required": ["query"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "calculate_trip_budget",
            "description": "Estimate a trip budget.",
            "parameters": {
                "type": "object",
                "properties": {
                    "no_of_days": {"type": "integer"},
                    "no_of_tourists": {"type": "integer"},
                    "flight_cost_per_person": {"type": "number"},
                    "hotel_cost_per_night": {"type": "number"},
                    "food_cost_per_day": {"type": "number"},
                    "activities_cost_per_day": {"type": "number"},
                    "misc_percent": {"type": "number", "default": 10.0},
                },
                "required": [
                    "no_of_days",
                    "no_of_tourists",
                    "flight_cost_per_person",
                    "hotel_cost_per_night",
                    "food_cost_per_day",
                    "activities_cost_per_day",
                ],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "search_documents",
            "description": "Search the user's uploaded travel PDF(s).",
            "parameters": {
                "type": "object",
                "properties": {"query": {"type": "string"}},
                "required": ["query"],
            },
        },
    },
]

def _history_to_messages(history):
    if not history:
        return []
    messages = []
    for turn in history:
        role = turn.get("role")
        content = turn.get("content")
        if role in ("user", "assistant") and content:
            messages.append({"role": role, "content": content})
    return messages

In [40]:
#RAG TOOL 

CHROMA_DIR = os.environ.get("chroma_directory", "chroma_db")
COLLECTION_NAME = "travel_docs"
CHUNK_SIZE = 800
CHUNK_OVERLAP = 100

_rag_client = None
_rag_collection = None


def _get_collection():
    global _rag_client, _rag_collection
    if _rag_collection is not None:
        return _rag_collection

    # Local, free embedding model (downloads once, runs on CPU, no API key needed).
    embed_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
        model_name="all-MiniLM-L6-v2"
    )
    _rag_client = chromadb.PersistentClient(path=CHROMA_DIR)
    _rag_collection = _rag_client.get_or_create_collection(
        name=COLLECTION_NAME,
        embedding_function=embed_fn,
    )
    return _rag_collection


def _extract_text(pdf_path: str) -> str:
    reader = PdfReader(pdf_path)
    pages = [page.extract_text() or "" for page in reader.pages]
    return "\n".join(pages)


def _chunk_text(text: str, chunk_size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP):
    text = " ".join(text.split())
    if not text:
        return []
    if overlap >= chunk_size:
        overlap = chunk_size // 4

    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks


def load_docs(pdf_path: str) -> int:
    """Extract, chunk, and index a PDF file. Returns the number of chunks stored."""
    if not pdf_path or not os.path.exists(pdf_path):
        raise FileNotFoundError(f"File not found: {pdf_path}")

    text = _extract_text(pdf_path)
    chunks = _chunk_text(text)
    if not chunks:
        raise ValueError("No extractable text found in this PDF (it may be scanned/image-only).")

    collection = _get_collection()
    base_name = os.path.splitext(os.path.basename(pdf_path))[0]
    ids = [f"{base_name}-{i}" for i in range(len(chunks))]
    metadatas = [{"source": os.path.basename(pdf_path), "chunk": i} for i in range(len(chunks))]

    collection.upsert(documents=chunks, ids=ids, metadatas=metadatas)
    return len(chunks)


def retrieve_docs(query: str, k: int = 4) -> List[str]:
    """Return the top-k most relevant chunks for a query. Empty list if nothing indexed."""
    if not query or not query.strip():
        return []
    collection = _get_collection()
    count = collection.count()
    if count == 0:
        return []
    results = collection.query(query_texts=[query], n_results=min(k, count))
    documents = results.get("documents", [[]])
    return documents[0] if documents else []


print("RAG functions ready (using local, free embeddings).")


RAG functions ready (using local, free embeddings).


In [41]:
# AGENT FUNCTION

def run_agent(message: str, history= None, max_steps: int = Max_Steps):
    client = _get_client()
    messages = [{"role": "system", "content": system_prompt}]
    messages.extend(_history_to_messages(history))
    messages.append({"role": "user", "content": message})

    for step in range(max_steps):
        
        # Ask LLM what to do
        response = client.chat.completions.create(model= _model, messages= messages, 
                                                  tools= tools_schema, tool_choice= "auto")
        msg = response.choices[0].message

        # If not tool is required, return final answer
        if not msg.tool_calls:
            return msg.content or "I don't have a response for that."

        messages.append(
            {"role": "assistant",
             "content": msg.content or "",
             "tool_calls": [tc.model_dump() for tc in msg.tool_calls]}
            )

        # If tool call exists, execute tool
        for tool_call in msg.tool_calls:
            
            # LLM selects the tool
            name = tool_call.function.name
            try:
                
                # Converts JSON arguments into dictionary
                args = json.loads(tool_call.function.arguments)
            
            except json.JSONDecodeError:
                args = {}
            
            func = tool_functions.get(name)
            
            if func is None:
                result = f"Unknown tool: {name}"
            
            else:
                try:
                    result = func(**args)
            
                except Exception as e:
                    result = f"Tool '{name}' raised an error: {e}"
            
            messages.append({"role": "tool", "tool_call_id": tool_call.id, "content": str(result)})

    return "I wasn't able to finish that request within the allowed number of steps."


print("Agent ready.")

Agent ready.


In [42]:
def upload_pdf(file):

    if file is None:
        return "No file uploaded."

    try:
        num_chunks = load_docs(file.name)
        return f"Indexed '{os.path.basename(file.name)}' ({num_chunks} chunks)."

    except FileNotFoundError:
        return "Upload failed: file not found on disk."

    except ValueError as e:
        return f"{e}"

    except Exception as e:
        return f"Failed to index file: {e}"


def chat(message, history):
    if not message or not message.strip():
        return "Please type a message."

    if not os.getenv("assg_groq_key"):
        return "API Key is not set."

    try:
        return run_agent(message, history=history)

    except Exception as e:
        return f"Something went wrong: {e}"


custom_css = """
:root {
    --accent-1: #2563eb;
    --accent-2: #06b6d4;
    --accent-soft: #eff6ff;
}

body[data-theme="ocean"]  { --accent-1: #2563eb; --accent-2: #06b6d4; --accent-soft: #eff6ff; }
body[data-theme="sunset"] { --accent-1: #f97316; --accent-2: #ec4899; --accent-soft: #fff7ed; }
body[data-theme="forest"] { --accent-1: #059669; --accent-2: #84cc16; --accent-soft: #ecfdf5; }
body[data-theme="grape"]  { --accent-1: #7c3aed; --accent-2: #d946ef; --accent-soft: #f5f3ff; }

* {
    box-sizing: border-box;
}

html, body {
    height: 100%;
    overflow-x: hidden;
}

.gradio-container {
    width: 100% !important;
    max-width: 1200px !important;
    margin: 0 auto !important;
    padding: 16px !important;
    font-family: 'Segoe UI', system-ui, sans-serif;
}

#hero-banner {
    border-radius: 16px;
    padding: 18px 22px;
    text-align: center;
    background: linear-gradient(135deg, var(--accent-1), var(--accent-2));
    box-shadow: 0 6px 18px rgba(0,0,0,0.12);
    margin-bottom: 12px;
    width: 100%;
}

#hero-banner h1 {
    color: #ffffff !important;
    font-size: 1.8em;
    font-weight: 800;
    margin: 0 0 4px 0;
    text-shadow: 0 2px 8px rgba(0,0,0,0.15);
}

#hero-banner p {
    color: rgba(255,255,255,0.92) !important;
    font-size: 0.92em;
    margin: 0;
}

#theme-picker {
    width: 100%;
    max-width: 420px;
    margin: 0 auto 14px auto !important;
    display: flex;
    justify-content: center;
}

#theme-picker > div {
    width: 100%;
}

/* Equal-height two-column layout that fills the viewport width */
#main-row {
    width: 100%;
    align-items: stretch !important;
    gap: 16px !important;
}

#sidebar-col, #chat-col {
    display: flex;
    flex-direction: column;
    min-width: 0;
}

#sidebar-card {
    background: var(--accent-soft);
    border: 1px solid rgba(0,0,0,0.06);
    border-radius: 14px;
    padding: 16px;
    box-shadow: 0 4px 14px rgba(0,0,0,0.05);
    height: 100%;
    display: flex;
    flex-direction: column;
}

#tools-card {
    background: #ffffff;
    border: 1px solid rgba(0,0,0,0.06);
    border-radius: 12px;
    padding: 12px 14px;
    margin-top: 12px;
    box-shadow: 0 2px 8px rgba(0,0,0,0.04);
}

.tool-badge-row {
    display: flex;
    flex-wrap: wrap;
    gap: 6px;
}

.tool-badge {
    display: inline-flex;
    align-items: center;
    gap: 6px;
    padding: 6px 12px;
    margin: 0;
    border-radius: 999px;
    font-size: 0.82em;
    font-weight: 600;
    color: white;
    white-space: nowrap;
    background: linear-gradient(90deg, var(--accent-1), var(--accent-2));
    box-shadow: 0 2px 6px rgba(0,0,0,0.1);
    transition: transform 0.15s ease, box-shadow 0.15s ease;
    cursor: default;
}

.tool-badge:hover {
    transform: translateY(-2px) scale(1.05);
    box-shadow: 0 6px 14px rgba(0,0,0,0.18);
}

#upload_status textarea {
    border-radius: 8px !important;
    width: 100% !important;
}

#pdf_input {
    width: 100% !important;
}

button.primary {
    background: linear-gradient(90deg, var(--accent-1), var(--accent-2)) !important;
    border: none !important;
    box-shadow: 0 3px 10px rgba(0,0,0,0.15);
    transition: transform 0.15s ease, box-shadow 0.15s ease;
}

button.primary:hover {
    transform: translateY(-1px) scale(1.03);
    box-shadow: 0 6px 16px rgba(0,0,0,0.2);
}

/* Make the chat column fill remaining height so it lines up with the sidebar */
#chat-col .chatbot {
    width: 100% !important;
}

footer { display: none !important; }

/* Stack columns on narrow / mobile widths instead of squeezing them */
@media (max-width: 900px) {
    #main-row {
        flex-direction: column !important;
    }
    #sidebar-col, #chat-col {
        width: 100% !important;
        flex: 1 1 100% !important;
    }
}
"""

set_theme_js = """
(theme) => {
    document.body.setAttribute('data-theme', theme);
    return theme;
}
"""

with gr.Blocks(title="Travel Planning Assistant", css=custom_css, theme=gr.themes.Soft(), fill_width=True) as demo:

    gr.HTML(
        """
        <div id="hero-banner">
            <h1>✈️ Travel-Planning Assistant</h1>
            <p>Ask about weather, currency, budgets, or upload a travel PDF to search.</p>
        </div>
        """
    )

    with gr.Row(elem_id="theme-picker"):
        theme_picker = gr.Radio(
            choices=["ocean", "sunset", "forest", "grape"],
            value="ocean",
            label="🎨 Color theme",
        )

    with gr.Row(elem_id="main-row", equal_height=False):

        with gr.Column(scale=1, min_width=280, elem_id="sidebar-col"):
            with gr.Column(elem_id="sidebar-card"):

                gr.Markdown("### 📄 Upload a Travel PDF")

                pdf_input = gr.File(label="PDF file", file_types=[".pdf"], elem_id="pdf_input")

                upload_status = gr.Textbox(label="Status", interactive=False, elem_id="upload_status")

                pdf_input.upload(upload_pdf, inputs=pdf_input, outputs=upload_status)

                with gr.Column(elem_id="tools-card"):
                    gr.Markdown("### 🛠 Available Tools")
                    gr.HTML(
                        """
                        <div class="tool-badge-row">
                            <span class="tool-badge">🌦 Weather</span>
                            <span class="tool-badge">💱 Currency</span>
                            <span class="tool-badge">🔎 Web Search</span>
                            <span class="tool-badge">💰 Budget Calculator</span>
                            <span class="tool-badge">📚 Document</span>
                        </div>
                        """
                    )

        with gr.Column(scale=2, min_width=400, elem_id="chat-col"):

            gr.ChatInterface(
                fn=chat,
                chatbot=gr.Chatbot(height=430, elem_classes=["chatbot"]),
                examples=[
                    "What is the weather in Rome?",
                    "Convert 200 USD to EUR",
                    "5-day trip to Paris for 2 people with flight $40/person, hotel $80/night, food $25/day, activities $40/day",
                ],
            )

    demo.load(fn=None, inputs=None, outputs=None, js="() => { document.body.setAttribute('data-theme', 'ocean'); }")
    theme_picker.change(fn=None, inputs=theme_picker, outputs=None, js=set_theme_js)

# Public URL
demo.launch(share=True, debug=True)

C:\Users\PMLS\AppData\Local\Temp\ipykernel_25464\2569353171.py:207: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(title="Travel Planning Assistant", css=custom_css, theme=gr.themes.Soft(), fill_width=True) as demo:


* Running on local URL:  http://127.0.0.1:7860

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


Keyboard interruption in main thread... closing server.
